# 🚀 Huấn luyện Vanilla Split Learning (CIFAR-10, ResNet-18) trên Google Colab

Notebook này hướng dẫn và hỗ trợ chạy toàn bộ pipeline **Vanilla Split Learning (SL)** Bước 0:
- **Mô hình**: ResNet-18 tùy chỉnh cho ảnh $32\times 32$ (CIFAR-10).
- **Phân tách Client - Server**:
  - **Client**: Stem (`conv1` + `bn1` + `relu`) + `layer1` $\rightarrow$ Smushed data $z \in \mathbb{R}^{64 \times 32 \times 32}$.
  - **Server**: `layer2` + `layer3` + `layer4` + `avgpool` + `fc` $\rightarrow$ Logits 10 lớp.
- **Tính năng nâng cao**:
  - 💾 **Lưu Checkpoint an toàn**: Tự động lưu `last_checkpoint.pt` sau mỗi epoch và `best_b0_vanilla.pt` khi đạt độ chính xác cao nhất.
  - 🔄 **Hỗ trợ Resume**: Khôi phục huấn luyện tiếp tục bất cứ lúc nào nếu Colab bị ngắt kết nối.
  - 📊 **Lưu Lịch sử**: Xuất ra cả file `history.json` và `history.csv` (train/test loss, acc, lr, epoch time).
  - 📈 **Vẽ Đồ thị trực quan**: Tự động sinh `training_curves.png` (Loss, Accuracy, Learning Rate, Time) và hiển thị trực tiếp.
  - ☁️ **Kết nối Google Drive**: Lưu trữ checkpoint vĩnh viễn trên Drive cá nhân.

---  
## 1. Kiểm tra Môi trường & GPU (Tesla T4 / V100 / A100)

In [ ]:
# Kiểm tra thông số GPU
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Bộ nhớ GPU (GB) : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Bạn đang chạy trên CPU! Hãy vào 'Runtime' -> 'Change runtime type' -> chọn 'T4 GPU' để train nhanh hơn.")

---  
## 2. Kết nối Google Drive (Lưu Checkpoint Vĩnh viễn)
> **Lợi ích**: Khi phiên Colab bị disconnect hoặc hết hạn, các checkpoint và lịch sử train đã lưu trong Google Drive vẫn được giữ nguyên để có thể tiếp tục train bằng `--resume`.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Thiết lập thư mục lưu checkpoint trên Drive
DRIVE_DIR = '/content/drive/MyDrive/AbReTAPE_Step0'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Đã kết nối Google Drive! Thư mục lưu checkpoint: {DRIVE_DIR}")

---  
## 3. Thiết lập Codebase (Clone hoặc Chuẩn bị Code)

In [ ]:
import os

# Clone repository nếu chưa có trong môi trường Colab:
if not os.path.exists('src'):
    !git clone https://github.com/CuongBien/AbReTAPE.git /content/AbReTAPE
    %cd /content/AbReTAPE
else:
    print("✅ Đã tìm thấy mã nguồn 'src'. Thư mục làm việc sẵn sàng!")

!ls -la


---  
## 4. Cài đặt Thư viện Phụ trợ & Tải Dataset CIFAR-10 Siêu tốc

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q -r requirements.txt


---  
## 5. Chạy Smoke Test (Kiểm thử đơn vị trước khi Train)

In [ ]:
# Kiểm thử lan truyền xuôi, ngược qua biên giới, evaluate và module vẽ đồ thị
!python run_tests.py


---  
## 6. Huấn luyện Vanilla Split Learning (100 Epochs)
- Checkpoint định kỳ, best model, file lịch sử và đồ thị sẽ được lưu trực tiếp vào Google Drive.

In [ ]:
# Chạy huấn luyện Split Learning với cấu hình lưu an toàn vào Google Drive
!python run_step0_vanilla.py \
    --epochs 100 \
    --batch-size 128 \
    --lr 0.1 \
    --eval-freq 1 \
    --data-dir data \
    --output-dir /content/drive/MyDrive/AbReTAPE_Step0


---  
## 7. Khôi phục Huấn luyện tiếp tục (Resume Training)
> **Dùng khi**: Colab bị disconnect hoặc muốn train tiếp từ epoch dở dang. Chỉ cần bật cờ `--resume`.

In [ ]:
# Khôi phục huấn luyện tiếp tục từ checkpoint gần nhất
!python run_step0_vanilla.py \
    --resume \
    --epochs 100 \
    --batch-size 128 \
    --lr 0.1 \
    --eval-freq 1 \
    --data-dir data \
    --output-dir /content/drive/MyDrive/AbReTAPE_Step0


---  
## 8. Hiển thị Đồ thị Trực quan (Loss, Accuracy, Learning Rate, Time)

In [ ]:
from IPython.display import Image, display
import os

plot_file = '/content/drive/MyDrive/AbReTAPE_Step0/training_curves.png'
if not os.path.exists(plot_file):
    plot_file = 'output/AbReTAPE_Step0/training_curves.png'

if os.path.exists(plot_file):
    print("📊 ĐỒ THỊ TRỰC QUAN HÓA QUÁ TRÌNH HUẤN LUYỆN:")
    display(Image(filename=plot_file, width=950))
else:
    print(f"⚠️ Chưa tìm thấy file đồ thị tại: {plot_file}")

---  
## 9. Thống kê & Đánh giá Lịch sử Huấn luyện (Dataframe)

In [ ]:
import json
import pandas as pd
import os

hist_file = '/content/drive/MyDrive/AbReTAPE_Step0/history.json'
if not os.path.exists(hist_file):
    hist_file = 'output/AbReTAPE_Step0/history.json'

if os.path.exists(hist_file):
    with open(hist_file, 'r', encoding='utf-8') as f:
        history = json.load(f)
    
    df = pd.DataFrame(history)
    df['train_acc'] = (df['train_acc'] * 100).round(2)
    if 'test_acc' in df and df['test_acc'].notna().any():
        df['test_acc'] = (df['test_acc'] * 100).round(2)
        best_idx = df['test_acc'].idxmax()
        best_row = df.loc[best_idx]
        print(f"🌟 [KẾT QUẢ TỐT NHẤT]")
        print(f"   - Epoch             : {int(best_row['epoch'])}")
        print(f"   - Best Test Accuracy: {best_row['test_acc']}%")
        print(f"   - Train Accuracy    : {best_row['train_acc']}%")
        print(f"   - Train Loss        : {best_row['train_loss']:.4f}")
        if 'test_loss' in best_row and pd.notna(best_row['test_loss']):
            print(f"   - Test Loss         : {best_row['test_loss']:.4f}")
        
        target_met = best_row['test_acc'] >= 92.0
        status = "ĐẠT CHUẨN (≥ 92%)" if target_met else "Chưa đạt 92% (cần train thêm epoch)"
        print(f"   - Tiêu chí nghiệm thu: {status}")
    
    print("\n📋 10 Epochs gần nhất:")
    display(df.tail(10))
else:
    print(f"⚠️ Không tìm thấy file lịch sử {hist_file}")

---  
## 10. (Tùy chọn) Chạy Đối chứng Centralized Training & Vẽ So sánh

In [ ]:
# Chạy Centralized Training nếu muốn đối chứng với Split Learning
# !python run_step0_vanilla.py --centralized \
#     --epochs 100 \
#     --batch-size 128 \
#     --lr 0.1 \
#     --eval-freq 1 \
#     --data-dir data \
#     --output-dir /content/drive/MyDrive/AbReTAPE_Step0


In [ ]:
# Vẽ biểu đồ so sánh Split Learning vs Centralized
import json
import os
import matplotlib.pyplot as plt

sl_hist_file = '/content/drive/MyDrive/AbReTAPE_Step0/history.json'
cent_hist_file = '/content/drive/MyDrive/AbReTAPE_Step0/centralized_history.json'

if os.path.exists(sl_hist_file) and os.path.exists(cent_hist_file):
    with open(sl_hist_file, 'r') as f:
        sl_data = json.load(f)
    with open(cent_hist_file, 'r') as f:
        cent_data = json.load(f)
        
    plt.figure(figsize=(14, 5))
    
    # Loss Comparison
    plt.subplot(1, 2, 1)
    plt.plot([x['epoch'] for x in sl_data], [x['train_loss'] for x in sl_data], label='Split Learning Loss', color='#1f77b4', linewidth=2)
    plt.plot([x['epoch'] for x in cent_data], [x['train_loss'] for x in cent_data], label='Centralized Loss', color='#ff7f0e', linestyle='--', linewidth=2)
    plt.title('So sánh Train Loss: Split Learning vs Centralized', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Accuracy Comparison
    plt.subplot(1, 2, 2)
    sl_eval = [x for x in sl_data if x.get('test_acc') is not None]
    cent_eval = [x for x in cent_data if x.get('test_acc') is not None]
    plt.plot([x['epoch'] for x in sl_eval], [x['test_acc']*100 for x in sl_eval], label='Split Learning Acc (%)', color='#2ca02c', linewidth=2)
    plt.plot([x['epoch'] for x in cent_eval], [x['test_acc']*100 for x in cent_eval], label='Centralized Acc (%)', color='#d62728', linestyle='--', linewidth=2)
    plt.title('So sánh Test Accuracy: Split Learning vs Centralized', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Test Accuracy (%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có đủ file history của cả 2 mô hình để vẽ so sánh.")